# Deep Learning - Winter Term 2026/2027

<hr style="border:2px solid gray">

### Exercise Sheet 01

<br>

## 3) Autograd Implementation [40 points]

In this task you build your own tiny autograd engine, similar to what PyTorch does under the hood when you call `loss.backward()`.

Each node in the computational graph is an object of the class `Node`, which stores its scalar value, its gradient and how to pass the gradient back to its inputs. Be careful when the same node is used more than once in a graph!

### 3(a) Forward and backward passes [30 points]

Complete the TODOs in the `Node` class: the forward passes of `sigmoid`, `tanh` and `relu`, and the backward passes of all nine operations (`+`, `*`, `**`, `exp`, `log`, `sqrt`, `sigmoid`, `tanh`, `relu`).

In [ ]:
# We only use NumPy for the computations
import numpy as np

In [ ]:
# Each Node object is one node of the computational graph.
class Node:

    ### Initialization of the class ###
    '''
    scalar: value of the node -- only scalars are used in this exercise
    _children: input nodes of the operation that created this node (a leading _ marks internal variables)
    _op: operation that created this node
    name: label used when printing/drawing the node
    '''
    def __init__(self, scalar, _children=(), _op='', name=''):
        self.scalar = scalar
        self.grad = 0.0 # By default we assume the variable does not affect the function
        self.name = name
        self._backward = lambda: None # Meaning the function doesn't do anything by default
        self._prev = set(_children) # Save the input nodes in a set.
        self._op = _op

    # This function just makes the printing of the Node human-readable
    def __repr__(self):
        return f"Node(name={self.name}, scalar={self.scalar}, grad={self.grad})"

    def __add__(self, other):
        other = other if isinstance(other, Node) else Node(other)
        out = Node(self.scalar + other.scalar, (self, other), '+') # forward pass

        def _backward():
            # d(out)/d(self) = 1 and d(out)/d(other) = 1,
            # so both inputs just receive out.grad. We use += (not =) because a node
            # can be used more than once, and all its gradients must be added up.
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward

        return out

    def __mul__(self, other):
        other = other if isinstance(other, Node) else Node(other)
        out = Node(self.scalar * other.scalar, (self, other), '*')

        def _backward():
            # d(out)/d(self) = other and d(out)/d(other) = self (product rule)
            self.grad += other.scalar * out.grad
            other.grad += self.scalar * out.grad
        out._backward = _backward

        return out

    def __pow__(self, other):
        # Only constant exponents (int/float) are supported, e.g. x**2 or x**-1
        assert isinstance(other, (int, float)), "only int/float powers are supported"
        out = Node(self.scalar**other, (self,), f'**{other}')

        def _backward():
            # power rule: d(x^n)/dx = n * x^(n-1)  (here n is called 'other')
            self.grad += other * self.scalar ** (other - 1) * out.grad
        out._backward = _backward

        return out

    def exp(self):
        x = self.scalar
        out = Node(np.exp(x), (self, ), 'exp')

        def _backward():
            # d(exp(x))/dx = exp(x), which is exactly the output value
            self.grad += out.scalar * out.grad
        out._backward = _backward

        return out

    def sigmoid(self):
        x = self.scalar
        out = Node(1 / (1 + np.exp(-x)), (self, ), 'sigmoid')

        def _backward():
            # d(sigmoid)/dx = sigmoid * (1 - sigmoid)  (that is Task 1a)
            self.grad += out.scalar * (1 - out.scalar) * out.grad
        out._backward = _backward

        return out

    def tanh(self):
        x = self.scalar
        out = Node(np.tanh(x), (self, ), 'tanh')

        def _backward():
            # d(tanh)/dx = 1 - tanh^2
            self.grad += (1 - out.scalar ** 2) * out.grad
        out._backward = _backward

        return out

    def relu(self):
        x = self.scalar
        out = Node(max(0.0, x), (self, ), 'relu')

        def _backward():
            # d(relu)/dx = 1 if x > 0, otherwise 0
            self.grad += (1.0 if self.scalar > 0 else 0.0) * out.grad
        out._backward = _backward

        return out

    def log(self):
        assert self.scalar > 0, "log is only defined for positive numbers"
        x = self.scalar
        out = Node(np.log(x), (self, ), 'log')

        def _backward():
            # d(ln(x))/dx = 1 / x
            self.grad += (1 / x) * out.grad
        out._backward = _backward

        return out

    def sqrt(self):
        assert self.scalar > 0, "No complex numbers here"
        x = self.scalar
        out = Node(np.sqrt(x), (self, ), 'sqrt')

        def _backward():
            # d(sqrt(x))/dx = 1 / (2 * sqrt(x)) = 0.5 / output
            self.grad += (0.5 / out.scalar) * out.grad
        out._backward = _backward

        return out

    # This function performs backpropagation for each node
    # Nothing to implement here but you can spend some time to understand how
    # the algorithm sorts the nodes.
    def backward(self):
        # topological order of all nodes in the graph (inputs before outputs)
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        # go one variable at a time and apply the chain rule to get its gradient
        # The gradient of the last operation wrt. itself is always 1.
        # We set this first gradient here.
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

    # Below are some necessary trivial definitions.
    # Feel free to use these functions. No gradient computation
    # is needed as they are implicitly implemented in prev. functions.
    # Below, we give an example of how this overloading works.
    def __neg__(self): # -self
        return self * -1

    def __radd__(self, other): # other + self
        return self + other

    def __sub__(self, other): # self - other
        return self + (-other)

    def __rsub__(self, other): # other - self
        return other + (-self)

    def __rmul__(self, other): # other * self
        return self * other

    def __truediv__(self, other): # self / other
        return self * other**-1

    def __rtruediv__(self, other): # other / self
        return other * self**-1

# ----------------------------------------------------------------
# Helper functions
# ----------------------------------------------------------------

def trace_nodes(node):
    '''
    Builds a list of all nodes of a graph. The method is
    only for inspection/debugging purposes. Feel free to use it if you need it.
    '''
    nodes = list()
    def build(v):
        if v not in nodes:
            nodes.insert(0, v)
            for child in (v._prev):
                build(child)
    build(node)
    return nodes

def print_graph(root):
    '''
    Produces a print out of the computational graph.
    '''
    nodes = trace_nodes(root)
    for n in nodes:
        print(n)

In [ ]:
# Here we give a graph visualization for your computational graph.
# You need networkx and matplotlib: pip install networkx matplotlib
# Feel free to play around with the visualization! For example, you can
# compare it with your own computational graph drawing.
import networkx as nx
import matplotlib.pyplot as plt

def draw_graph(root):
    # one graph node per Node, with an edge from each input to the output of an operation
    G = nx.DiGraph()
    for n in trace_nodes(root):
        op = f" ({n._op})" if n._op else ""
        G.add_node(id(n), label=f"{f'{n.name}{op}'.strip()}\nscalar={round(n.scalar, 4)}\ngrad={round(n.grad, 4)}")
        for child in n._prev:
            G.add_edge(id(child), id(n))

    # assign each node to a layer by its longest distance to the root, so the root
    # is on the right and every node is close to its output. Inside a layer the
    # nodes are sorted by their label (i.e. name) to get a reproducible drawing.
    layers = list(reversed(list(nx.topological_generations(G.reverse()))))
    pos = {}
    for x, layer in enumerate(layers):
        for y, v in enumerate(sorted(layer, key=lambda v: G.nodes[v]["label"])):
            pos[v] = (x, (len(layer) - 1) / 2 - y)

    # the figure grows with the number of layers and the size of the largest layer
    plt.figure(figsize=(2.6 * len(layers), 1.4 * max(len(l) for l in layers) + 1))
    nx.draw_networkx_edges(G, pos, node_size=6000, arrowsize=15, edge_color="gray")
    nx.draw_networkx_labels(G, pos, labels=nx.get_node_attributes(G, "label"), font_size=9,
                            bbox=dict(boxstyle="round", fc="lightyellow", ec="black"))
    plt.axis("off")
    plt.show()

### Example

There are no automatic tests: verify your implementation yourself by comparing it with gradients you computed by hand. You can look at a gradient by printing either the node itself or `node.grad`.

Let's start with a tiny equation:
$$
\mathcal{Z} = a \cdot b + c
$$

In [ ]:
# Example: build Z, run backward() and print the gradients.
a = Node(2.0,  name='a')
b = Node(-3.0, name='b')
c = Node(10.0, name='c')

# Due to overloading the operators we can use the arithmetic
# operators like +, * directly in expressions. Note that the
# expression
#
#     a * b + c
#
# is equivalent to
#
#     a * b + c = a.__mul__(b).__add__(c)
#
# Moreover, it automatically creates intermediate nodes for
# a * b and _ + c.
#
# In order to inspect every single piece of the graph, we
# name the intermediate nodes:

Z_t1 = a * b; Z_t1.name='Z_t1'
Z_t2 = Z_t1 + c; Z_t2.name='Z_t2'

# We run the backward function to compute all the gradients starting from the end node.
# The gradient of each node n is stored in n.grad
Z_t2.backward()

# Print the nodes to see their name, scalar and *most importantly* their gradients.
print(c)
print(a)
print(b)
print(Z_t1)
print(Z_t2)
print()

# We could have written this more compact (as explained above):
a.grad=0; b.grad=0; c.grad=0
Z = a * b + c; Z.name='Z'
Z.backward()
print_graph(Z)
#
# Notes:
# - Due to the use of the data structure set, the nodes stored in
#   every _prev variable are not arranged in the order they have been
#   inserted -> this is why print_graph may print them in a different
#   order (don't worry about it).
# - A repeated call of backward() will cause an accumulation of gradients
#   within the respective nodes (this is why we reset the inputs to 0
#   above and build a new graph).

In [ ]:
# With the following one-liner, you can visualize the comp. graph
draw_graph(Z_t2)

### 3(b) Test your implementation on $\mathcal{A}$ [10 points]

Build the equation from Task 2 with your engine:
$$
\mathcal{A} = \sqrt{a + b + c^2} + \ln(a + b + c^2) + \frac{a + b + c^2}{bc^2}
$$
Create one named node per intermediate variable from Task 2(a), use the input values from Task 2(d), run `backward()` and compare all scalars and gradients with your hand computation.

**Note:** `x / y` is computed as `x * y**-1`, so your graph contains an extra unnamed `**-1` node.

In [ ]:
# --- Build A with one named node per intermediate variable (Task 2a) ---
# Input values from Task 2(d)
a = Node(1.0, name='a')
b = Node(2.0, name='b')
c = Node(1.0, name='c')

t1 = c ** 2;        t1.name = 't1'   # t1 = c^2
t2 = a + b;         t2.name = 't2'   # t2 = a + b
t3 = t2 + t1;       t3.name = 't3'   # t3 = t2 + t1
t4 = b * t1;        t4.name = 't4'   # t4 = b * t1
t5 = t3.log();      t5.name = 't5'   # t5 = ln(t3)
t6 = t3.sqrt();     t6.name = 't6'   # t6 = sqrt(t3)
t7 = t3 / t4;       t7.name = 't7'   # t7 = t3 / t4  (creates an extra unnamed **-1 node)
t8 = t5 + t7;       t8.name = 't8'   # t8 = t5 + t7
A  = t6 + t8;       A.name  = 'A'    # A  = t6 + t8

# Backward pass: computes dA/dv for every node v
A.backward()

# --- Compare with the hand computation from Task 2(d) ---
# name: (node, value by hand, gradient by hand)
hand = {
    'a':  (a,  1,     1),
    'b':  (b,  2,     0),
    'c':  (c,  1,    -2),
    't1': (t1, 1,    -1),
    't2': (t2, 3,     1),
    't3': (t3, 4,     1),
    't4': (t4, 2,    -1),
    't5': (t5, 1.386, 1),
    't6': (t6, 2,     1),
    't7': (t7, 2,     1),
    't8': (t8, 3.386, 1),
    'A':  (A,  5.386, 1),
}

print(f"{'node':<5}{'scalar':>10}{'by hand':>10}{'grad':>10}{'by hand':>10}   match?")
for name, (node, value_hand, grad_hand) in hand.items():
    ok = np.isclose(node.scalar, value_hand, atol=1e-3) and np.isclose(node.grad, grad_hand, atol=1e-3)
    print(f"{name:<5}{node.scalar:>10.4f}{value_hand:>10}{node.grad:>10.4f}{grad_hand:>10}   {'OK' if ok else 'DIFFERENT'}")

# Draw the graph (the unnamed nodes are the extra ones created by x / y)
draw_graph(A)


***Optional tip:*** You can check any gradient numerically with the finite difference $\frac{f(x+h) - f(x)}{h}$ for a small $h$, e.g. $h = 10^{-6}$. This is how a derivative is mathematically defined:
```python
f = lambda x: np.sqrt(x)
h = 1e-6
print((f(4.0 + h) - f(4.0)) / h)  # approx. 0.25 = 0.5 / sqrt(4)
```